In [2]:
import json
from pathlib import Path


# Arquivo JSON de entrada
arquivo_entrada = Path("organizacao_com_exercicios.json")

# Arquivo JSON de saída
arquivo_saida = Path("organizacao_final.json")


# Campos antigos que serão removidos após a criação dos exercícios
CAMPOS_EXEMPLOS = [
    "exemplo_adicional",
    "exemplo_diferente",
    "exemplo_episodio"
]


def separar_frase_traducao(texto):
    """
    Separa a frase em inglês da tradução em português.

    Formato esperado:
    English sentence (Tradução em português)

    Retorna:
    (frase, tradução)
    """

    if not isinstance(texto, str) or not texto.strip():
        return "", ""

    texto = texto.strip()

    # Usa o último parêntese de abertura para separar a tradução.
    # Isso permite que a frase em inglês contenha parênteses.
    posicao = texto.rfind(" (")

    if posicao == -1 or not texto.endswith(")"):
        return texto, ""

    frase = texto[:posicao].strip()
    traducao = texto[posicao + 2:-1].strip()

    return frase, traducao


def criar_exercicios(item, categoria, indice):
    """
    Cria exercícios para um item que ainda não possui exercícios.

    As alternativas e a resposta correta ficam vazias
    para preenchimento posterior.
    """

    exercicios = []

    exemplos = [
        ("exemplo_episodio", "episodio", True),
        ("exemplo_adicional", "adicional", False),
        ("exemplo_diferente", "diferente", False)
    ]

    palavra = item.get("palavra", "")

    for numero, (campo, origem, eh_exemplo) in enumerate(
        exemplos, start=1
    ):

        texto_exemplo = item.get(campo, "")

        frase, traducao = separar_frase_traducao(texto_exemplo)

        # Não cria exercícios para exemplos inexistentes ou vazios.
        if not frase:
            continue

        exercicio = {
            "id": f"{categoria[:3]}_{indice:03d}_ex{numero}",
            "frase": frase,
            "foco": palavra,
            "origem_exemplo": origem,
            "eh_exemplo_episodio": eh_exemplo,
            "alternativas": [
                {"id": "a", "texto": ""},
                {"id": "b", "texto": ""},
                {"id": "c", "texto": ""},
                {"id": "d", "texto": ""}
            ],
            "resposta_correta": "",
            "traducao_correta": traducao
        }

        exercicios.append(exercicio)

    return exercicios


def processar_json(dados):
    """
    Primeiro cria exercícios para itens que ainda não os possuem.
    Depois remove os campos antigos de exemplos de todo o JSON.
    """

    total_itens = 0
    total_exercicios_criados = 0
    total_exercicios_preservados = 0
    total_campos_removidos = 0

    for episodio in dados.get("episodios", []):

        for categoria, itens in episodio.items():

            # Ignora propriedades do episódio que não sejam listas
            # de itens de estudo.
            if not isinstance(itens, list):
                continue

            for indice, item in enumerate(itens, start=1):

                if not isinstance(item, dict):
                    continue

                total_itens += 1

                # ETAPA 1:
                # Cria os exercícios somente se ainda não existirem.
                if "exercicios" not in item:
                    exercicios = criar_exercicios(
                        item,
                        categoria,
                        indice
                    )

                    item["exercicios"] = exercicios
                    total_exercicios_criados += len(exercicios)

                else:
                    # Preserva integralmente os exercícios existentes.
                    total_exercicios_preservados += len(
                        item["exercicios"]
                    )

                # ETAPA 2:
                # Remove os campos antigos somente depois
                # de garantir a existência da estrutura de exercícios.
                for campo in CAMPOS_EXEMPLOS:
                    if campo in item:
                        del item[campo]
                        total_campos_removidos += 1

    return {
        "total_itens": total_itens,
        "total_exercicios_criados": total_exercicios_criados,
        "total_exercicios_preservados": total_exercicios_preservados,
        "total_campos_removidos": total_campos_removidos
    }


def main():
    if not arquivo_entrada.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {arquivo_entrada.resolve()}"
        )

    # Carrega o JSON original.
    with arquivo_entrada.open("r", encoding="utf-8") as arquivo:
        dados = json.load(arquivo)

    # Processa todos os episódios.
    estatisticas = processar_json(dados)

    # Salva em outro arquivo, preservando o original.
    with arquivo_saida.open("w", encoding="utf-8") as arquivo:
        json.dump(
            dados,
            arquivo,
            ensure_ascii=False,
            indent=2
        )

    print("Processamento concluído!")
    print(f"Arquivo gerado: {arquivo_saida.resolve()}")
    print(f"Itens processados: {estatisticas['total_itens']}")
    print(
        "Exercícios criados: "
        f"{estatisticas['total_exercicios_criados']}"
    )
    print(
        "Exercícios existentes preservados: "
        f"{estatisticas['total_exercicios_preservados']}"
    )
    print(
        "Campos antigos removidos: "
        f"{estatisticas['total_campos_removidos']}"
    )


if __name__ == "__main__":
    main()

Processamento concluído!
Arquivo gerado: C:\Users\ronal\source\repos\EnglishRoSa\dados\organizacao_final.json
Itens processados: 628
Exercícios criados: 1590
Exercícios existentes preservados: 294
Campos antigos removidos: 1884
